# 06. Governance, Guardrails & Hierarchy Fallback

Covers **Attributes 13, 16, 18, 21, 22, 24, 25**:
- Hierarchy-aware fallback (city -> country roll-up)
- Unsupported entity and competitor boundary detection
- Metadata discovery & schema queries
- Context-aware follow-up suggestions
- Transparent reporting of assumptions & system limitations
- Numeric overlap verification and retry mechanisms

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if "high_level" in str(pathlib.Path.cwd()) or "capabilities" in str(pathlib.Path.cwd()) else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.orchestrator import Orchestrator
from src.llm_client import MockLLMClient
from src.tools.sql_tool import run_query, validate_sql
from src.tools.retrieval_tool import get_index
from src.tools.code_tool import run_code
from src.formatting import format_value, rows_to_markdown_table

print("AB InBev Enterprise Q&A Agent Pipeline Loaded.")

AB InBev Enterprise Q&A Agent Pipeline Loaded.

### 1. Hierarchy-Aware Fallback (City to Country)

In [2]:
orch = Orchestrator(llm_router=MockLLMClient(), llm_worker=MockLLMClient())
cities = ["St. Louis", "Monterrey", "Leuven", "Brussels", "Shanghai", "Mumbai", "Sao Paulo"]
for city in cities:
    r = orch.handle_turn(f"How did brands perform in {city} in 2025?")
    print(f"City: {city} -> Country SQL: {r.sql_used}")
    print(f"Assumption Note: {r.assumptions[0] if r.assumptions else 'None'}\n")

City: St. Louis -> Country SQL: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand='Corona' AND country='United States' AND year=2025 GROUP BY brand, country, year LIMIT 500
Assumption Note: Structured data isn't broken out by city; showing **United States** (the country containing St. Louis) instead.

City: Monterrey -> Country SQL: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM fact_monthly_kpi WHERE brand='Corona' AND country='Mexico' AND year=2025 GROUP BY brand, country, year LIMIT 500
Assumption Note: Structured data isn't broken out by city; showing **Mexico** (the country containing Monterrey) instead.

City: Leuven -> Country SQL: SELECT brand, country, year, SUM(net_revenue_usd) AS net_revenue_usd, SUM(volume) AS volume, AVG(market_share_pct) AS market_share_pct FROM f

### 2. Unsupported Entity & Competitor Disclosures

In [3]:
competitors = ["Heineken", "Carlsberg", "Molson Coors"]
for comp in competitors:
    r = orch.handle_turn(f"How is {comp} doing in Europe?")
    print(f"Query for {comp}:\nAssumptions surfaced: {r.assumptions}\n")

Query for Heineken:
Assumptions surfaced: ["'Heineken' isn't part of Anheuser-Busch InBev's tracked entities (brand/country/competitor), so no internal data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting."]

Query for Carlsberg:
Assumptions surfaced: ["'Carlsberg' isn't part of Anheuser-Busch InBev's tracked entities (brand/country/competitor), so no internal data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting."]

Query for Molson Coors:
Assumptions surfaced: ["'Molson Coors' isn't part of Anheuser-Busch InBev's tracked entities (brand/country/competitor), so no internal data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting."]


### 3. Metadata Discovery & Context-Aware Suggestions

In [4]:
r_meta = orch.handle_turn("What data is available in the catalog?")
print("Metadata Catalog Answer:\n" + r_meta.answer[:300] + "...\n")

r_sug = orch.handle_turn("What was Budweiser revenue in US in 2025?")
print("Follow-up Suggestions generated:")
for s in r_sug.follow_up_suggestions:
    print(f"- {s}")

Metadata Catalog Answer:
**Available data** (Jan 2023–Aug 2026):

KPIs: Net Revenue (USD), Volume (hL), Market Share (%), Net Revenue per hL (ASP) (USD per hL), Distribution (ACV / BEES Reach) (%), Marketing Spend (USD), Promotion Spend (USD), Gross Margin (%)

Brands & categories:
- Premium & Above (Global Premium, Premium...

Follow-up Suggestions generated:
- Compare against Volume?
- Compare this to the same period last year?